# 第一階段：數據管理與基礎預處理 (Data Pipeline)

本階段建立數據加載與清理流程，採用 SQLite 存儲歷史市場數據，並執行對數轉換與均值回歸基礎檢驗。

### 參考文獻：
- Engle, R. F., & Granger, C. W. (1987). *Co-integration and error correction*. Econometrica.

In [1]:
import subprocess, sys
for pkg in ['statsmodels','plotly','ipywidgets','kaleido','scikit-learn','yfinance']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])
import warnings; warnings.filterwarnings('ignore')
import sqlite3, numpy as np, pandas as pd
import logging
from itertools import combinations
from statsmodels.tsa.stattools import coint
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML
pd.set_option('display.float_format','{:.4f}'.format)
import yfinance as yf
import time
import os
# print('套件導入完成。')

In [2]:
import os as _os

# ==========================================
# 🛑 全局模式切換開關 (True: 快速開發測試 / False: 論文最終完整回測)
# ==========================================
FAST_TEST_MODE = False

if FAST_TEST_MODE:
    print("🚀 啟動【快速測試模式】: 僅回測 2019-2022 年間之科技板塊。")
    # 縮短時間：涵蓋 2020 疫情崩盤與 2022 升息的壓力測試區間
    START_DATE = '2019-01-01'
    END_DATE = '2022-12-31'
    # 縮小股票池：僅測試單一板塊，運算量大幅減少
    TARGET_SECTOR = 'Information Technology'  
else:
    print("🐢 啟動【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。")
    # 論文要求的全樣本時間
    START_DATE = '2000-01-01'
    END_DATE = '2025-12-31'
    # 測試全市場 (設為 None 代表不限制單一產業)
    TARGET_SECTOR = None  

# ==========================================
# 📊 共用策略參數 (兩種模式皆適用)
# ==========================================
DB_PATH           = r'..\data\sp500_data.db'
# 產業動態補齊開關與快取路徑
USE_DYNAMIC_SECTORS = True 
IMPUTED_SECTOR_PATH = r'..\data\imputed_sectors.csv'

# 視窗滾動參數
FORMATION_WINDOW = 252   # 形成期 (約一年)
TRADING_WINDOW = 126     # 交易期 (約半年)
ROLLING_WINDOW = 20      # 滾動步長 (約一個月)
MIN_HISTORY_DAYS  = 200

# 配對與統計檢定參數
TOP_N_PAIRS = [1, 3]    # 
COINT_P_VALUE = 0.01     # 嚴格的共整合 p-value 門檻
SECTOR_NEUTRAL = True    # 是否限制同產業配對 (當 TARGET_SECTOR 為 None 時生效)

# 交易執行與資金控管參數
Z_ENTRY = 2            # 進場門檻
Z_EXIT = 0.0             # 出場門檻 (均值回歸)
MAX_LOSS_PCT = 3     # 嚴格的單筆未實現損益停損 (5%)
TRANSACTION_COST = 0.0029 # 雙邊交易手續費 (0.29%)
HEDGE_RATIO = 1.0        # 避險比例 (後續已被動態 Beta 取代，此處保留作為備用)

INITIAL_CAPITAL = 10000  # 初始本金
CAPITAL_TRANCHES = 10       # 滾動視窗資金切割份數

# 確保快取檔案的上一層資料夾存在
os.makedirs(os.path.dirname(IMPUTED_SECTOR_PATH), exist_ok=True)

# 自動轉換單一數值為 List 以防止後續 Iterable 報錯
if isinstance(TOP_N_PAIRS, int):
    TOP_N_PAIRS = [TOP_N_PAIRS]

🐢 啟動【完整回測模式】: 回測 2000-2025 年間之全市場 S&P 500。


In [3]:
def load_data_from_db(db_path, start_date, end_date):
    """Load price and sector data from SQLite."""
    conn = sqlite3.connect(db_path)
    price_queries = [
        (f"SELECT date, ticker, open, high, low, adj_close AS close, volume FROM daily_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
        (f"SELECT date, ticker, open, high, low, close, volume FROM stock_prices "
         f"WHERE date BETWEEN '{start_date}' AND '{end_date}' ORDER BY date"),
    ]
    prices_df = None
    for q in price_queries:
        try:
            prices_df = pd.read_sql_query(q, conn, parse_dates=['date'])
            if len(prices_df) > 0:
                print(f'Price rows: {len(prices_df):,}')
                break
        except Exception:
            continue
    if prices_df is None or len(prices_df) == 0:
        tbls = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
        conn.close()
        raise RuntimeError(f"No price table. Tables: {tbls['name'].tolist()}")
    sector_queries = [
        "SELECT ticker, sector FROM tickers",
        "SELECT ticker, sector FROM sp500_components GROUP BY ticker",
    ]
    sector_df = None
    for q in sector_queries:
        try:
            sector_df = pd.read_sql_query(q, conn)
            if len(sector_df) > 0:
                print(f'Sector rows: {len(sector_df):,}')
                break
        except Exception:
            continue
    conn.close()
    if sector_df is None or len(sector_df) == 0:
        print('WARNING: No sector table, using Unknown.')
        sector_df = pd.DataFrame({'ticker': prices_df['ticker'].unique(), 'sector': 'Unknown'})
        
    # [Mod] Mitigate Survivorship Bias: Check if delisted components exist
    max_date = prices_df['date'].max()
    max_dates = prices_df.groupby('ticker')['date'].max()
    delisted_count = (max_dates < max_date - pd.Timedelta(days=30)).sum()
    if delisted_count < 10:
        import logging
        logging.warning("STRICT RESEARCH LIMITATION: Database lacks historically delisted S&P 500 components. Survivorship Bias is present.")
        
    return prices_df, sector_df



def fix_unknown_sectors(sector_df, use_dynamic=True, save_path=r'data\imputed_sectors.csv'):
    """具備本機快取與全域開關控制的產業補齊模組"""
    # 1. 開關判斷：如果不使用動態補齊，直接原封不動回傳
    if not use_dynamic:
        print("不使用動態產業補齊，維持原始 Unknown 分類作為對照組。")
        return sector_df

    # 確保儲存的目錄 (data\) 存在
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # 2. 快取讀取：如果已經抓過並存檔，直接載入
    if os.path.exists(save_path):
        print(f"從本機快取載入已補齊的產業分類: {save_path}")
        cached_df = pd.read_csv(save_path)
        update_df = cached_df.set_index('ticker')
        sector_df = sector_df.set_index('ticker')
        sector_df.update(update_df)
        return sector_df.reset_index()

    # 3. API 抓取：如果沒有快取，執行連線作業
    unknown_mask = sector_df['sector'] == 'Unknown'
    unknown_tickers = sector_df[unknown_mask]['ticker'].tolist()
    
    if not unknown_tickers:
        return sector_df

    print(f"找不到本機快取，正在透過 API 補齊 {len(unknown_tickers)} 檔 Unknown 股票的產業分類...")
    
    yf_logger = logging.getLogger('yfinance')
    original_level = yf_logger.level
    yf_logger.setLevel(logging.CRITICAL) 
    
    fixed_sectors = []
    
    for i, ticker in enumerate(unknown_tickers):
        try:
            info = yf.Ticker(ticker).info
            sector = info.get('sector', 'Unknown')
            fixed_sectors.append({'ticker': ticker, 'sector': sector})
            time.sleep(0.02) 
        except Exception:
            fixed_sectors.append({'ticker': ticker, 'sector': 'Unknown'})
            
        if (i + 1) % 50 == 0:
            print(f"已處理 {i + 1} / {len(unknown_tickers)}...")
            
    yf_logger.setLevel(original_level)
    
    # 4. 儲存快取：將剛抓下來的資料存成 CSV，下次就不用再抓了
    fetched_df = pd.DataFrame(fixed_sectors)
    fetched_df.to_csv(save_path, index=False)
    print(f"API 抓取完畢！已將動態產業分類永久儲存至: {save_path}")
    
    # 更新回原本的 DataFrame
    update_df = fetched_df.set_index('ticker')
    sector_df = sector_df.set_index('ticker')
    sector_df.update(update_df)
    sector_df = sector_df.reset_index()
    
    remaining = len(sector_df[sector_df['sector'] == 'Unknown'])
    print(f"補齊完成！剩餘真實無法識別(已下市)的 Unknown 股票數量: {remaining}")
    
    return sector_df

def preprocess_prices(prices_df, min_days=MIN_HISTORY_DAYS):
    """Pivot, forward-fill, drop sparse tickers."""
    pivot = prices_df.pivot_table(index='date', columns='ticker', values='close', aggfunc='last')
    pivot.index = pd.to_datetime(pivot.index)
    pivot.sort_index(inplace=True)
    pivot.ffill(limit=5, inplace=True)
    valid = pivot.columns[pivot.notna().sum() >= min_days]
    pivot = pivot[valid]
    print(f'Matrix: {len(pivot)} days x {len(pivot.columns)} tickers')

    # [Mod] Integrate Macroeconomic Indicator (VIX)
    import yfinance as yf
    print("Fetching VIX data...")
    try:
        vix_data = yf.download("^VIX", start=pivot.index.min(), end=pivot.index.max() + pd.Timedelta(days=1), progress=False)
        if isinstance(vix_data.columns, pd.MultiIndex):
            vix_close = vix_data['Close'].squeeze()
        else:
            vix_close = vix_data['Close']
        vix_df = pd.DataFrame({'VIX': vix_close})
        vix_df.index = pd.to_datetime(vix_df.index).tz_localize(None)
        # [Mod] Shift VIX by 1 day to completely prevent look-ahead bias
        vix_shifted = vix_df.shift(1)
        vix_aligned = vix_shifted.reindex(pivot.index).ffill()
        print("VIX feature matrix aligned.")
    except Exception as e:
        print(f"Error fetching VIX: {e}")
        vix_aligned = pd.DataFrame(index=pivot.index, columns=['VIX'])

    return pivot, vix_aligned


In [4]:
# 1. 從資料庫載入原始資料
prices_raw, sector_info = load_data_from_db(DB_PATH, START_DATE, END_DATE)

# 2. 攔截並補齊 Unknown 產業分類 (傳入全域開關與路徑)
sector_info = fix_unknown_sectors(
    sector_info, 
    use_dynamic=USE_DYNAMIC_SECTORS, 
    save_path=IMPUTED_SECTOR_PATH
)

# 3. 執行原有的價格矩陣轉換與 VIX 融合
price_pivot, vix_features = preprocess_prices(prices_raw)
sector_map = sector_info.set_index('ticker')['sector'].to_dict()

# 4. 顯示結果
print(pd.Series(sector_map).value_counts().head(10))

Price rows: 3,607,121
Sector rows: 843
從本機快取載入已補齊的產業分類: ..\data\imputed_sectors.csv
Matrix: 6539 days x 660 tickers
Fetching VIX data...
VIX feature matrix aligned.
Unknown                   214
Industrials                94
Financials                 76
Information Technology     71
Health Care                60
Consumer Discretionary     48
Consumer Cyclical          37
Real Estate                36
Consumer Staples           36
Energy                     35
Name: count, dtype: int64


# 第二階段：動態配對篩選與特徵工程 (Pair Selection & Feature Engineering)

採用 PCA 降維與 HDBSCAN 聚類篩選具備相似統計行為的股票池，減少 RL 的狀態空間維度。

### 參考文獻：
- Campagnoli, T., et al. (2023). *Dynamic pairs trading using clustering and reinforcement learning*.

In [5]:
from joblib import Parallel, delayed
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm


import numpy as np
import statsmodels.api as sm

def compute_hurst(ts):
    if len(ts) < 20: return 0.5
    lags = range(2, 20)
    # 確保不會遇到負數或零的警告
    with np.errstate(invalid='ignore', divide='ignore'):
        var_diff = [np.var(ts[lag:] - ts[:-lag]) for lag in lags]
    
    valid = [v > 0 and not np.isnan(v) for v in var_diff]
    if sum(valid) < 5: return 0.5
    
    poly = np.polyfit(np.log(np.array(lags)[valid]), np.log(np.array(var_diff)[valid]), 1)
    return poly[0] / 2.0
    
def compute_half_life(ts):
    z_lag = np.roll(ts, 1)
    z_lag[0] = 0
    z_ret = ts - z_lag
    z_ret[0] = 0
    z_lag2 = sm.add_constant(z_lag)
    try:
        res = sm.OLS(z_ret[1:], z_lag2[1:]).fit()
        hl = -np.log(2) / res.params[1]
        return hl if (hl > 0 and hl < 100) else 15.0
    except:
        return 15.0

def extract_micro_ts_features(ts):
    """提取微觀特徵，並確保回傳值數量與 DataFrame 欄位一致"""
    ticker = ts.name
    try:
        ts_vals = ts.dropna()
        if len(ts_vals) < 30:
            return tuple([ticker] + [np.nan] * 7) # 回傳 8 個元素
            
        # 1. 最近期對數報酬率
        log_ret = np.log(ts_vals / ts_vals.shift(1)).dropna()
        log_ret_latest = log_ret.iloc[-1]
        
        # 2. 波動率 (近似 ATR 概念：High-Low 或純粹標準差)
        atr_latest = ts_vals.rolling(14).std().iloc[-1]
        
        # 3. Z-Score (相對於 60 天均值)
        roll_mean = ts_vals.rolling(60).mean()
        roll_std = ts_vals.rolling(60).std()
        z_score_latest = ((ts_vals - roll_mean) / roll_std).iloc[-1]
        
        # 4. 成交量失衡 (假設目前無 Volume 欄位，先填 0 或結合 VIX)
        vol_imb_latest = 0.0 
        
        # 5. Hurst Exponent
        hurst = compute_hurst(ts_vals.values)
        
        # 6. Half-life (均值回歸半衰期)
        half_life = compute_half_life(ts_vals.values)
        
        # 7. ADF 檢定統計量
        try:
            adf_res = adfuller(ts_vals.values, maxlag=1, regression='c', autolag=None)
            adf_stat = adf_res[0]
        except:
            adf_stat = np.nan
            
        # [修復點] 完整回傳 8 個變數，對應特徵矩陣的欄位
        return ticker, log_ret_latest, atr_latest, z_score_latest, vol_imb_latest, hurst, half_life, adf_stat
        
    except Exception as e:
        # 發生錯誤時也必須回傳 8 個元素，避免 unpack 失敗
        return tuple([ticker] + [np.nan] * 7)

def feature_engineering_pipeline(panel_df, vix_series, n_jobs=-1, n_pca_components=3, scaler_in=None):
    """
    建構橫斷面特徵矩陣，直接準備好輸出給 DBSCAN 使用
    panel_df: 含有 open, high, low, close, volume 的 DataFrame
    vix_series: 此觀測窗口的 VIX 時間序列
    """
    print("啟動特徵工程 Pipeline...")
    
    # 確認 panel_df 是以 [date, ticker] 或是可以直接 pivot
    if not isinstance(panel_df.index, pd.MultiIndex):
        panel_df = panel_df.set_index(['date', 'ticker'])
        
    close_pivot = panel_df['close'].unstack(level='ticker')
    log_returns = np.log(close_pivot / close_pivot.shift(1)).fillna(0)
    
    # A. 將 Close 價格拉平計算全市場因子 (PCA) 與 相關性
    pca = PCA(n_components=n_pca_components)
    market_factors = pca.fit_transform(log_returns)
    reconstructed = pca.inverse_transform(market_factors)
    
    # 提取特有殘差 (PCA Residuals)
    pca_residuals = log_returns - reconstructed
    pca_res_latest = pca_residuals.iloc[-1]
    
    # 動態 Pearson Correlation (與 PCA 第一主成分計算短週期的共同波動性)
    mf_1_series = pd.Series(market_factors[:, 0], index=log_returns.index)
    market_corr = log_returns.apply(lambda x: x.iloc[-20:].corr(mf_1_series.iloc[-20:]))
    
    # B. 分配平行運算：迴圈切割給 CPU 計算各檔股票的微觀高強度特徵
    tickers = close_pivot.columns
    print(f"啟動多執行緒處理 {len(tickers)} 檔股票之時間序列與 ADF 微觀特徵...")
    
    # 將每個 ticker 的歷史 df 提早整理出來給 parallel
    # 避免在 function 裡面重新 filter 導致效率極差
    ticker_dfs = {t: panel_df.xs(t, level='ticker') for t in tickers if t in panel_df.index.get_level_values('ticker')}
    valid_tickers = list(ticker_dfs.keys())

    results = Parallel(n_jobs=n_jobs)(
        delayed(extract_micro_ts_features)(ticker_dfs[t]['close'].rename(t)) for t in valid_tickers
    )
    
    cols = ['ticker', 'Log_Ret', 'ATR', 'Z_Score', 'Vol_Imbalance', 'Hurst', 'Half_life', 'ADF_stat']
    features_df = pd.DataFrame([r for r in results if len(r) == 8], columns=cols).set_index('ticker') # 確保回傳長度相符
    features_df.columns = cols[1:]
    
    # C. 合併市場與總體經濟層面的特徵 (Macro Awareness)
    features_df['PCA_Res'] = pca_res_latest
    features_df['Market_Corr'] = market_corr
    features_df['VIX'] = vix_series.iloc[-1] if not vix_series.empty else np.nan
    
    features_df = features_df.dropna() # 清理計算中斷不穩定的死點
    
    # D. 嚴格的特徵標準化機制 (Standardization / Z-Score Scaling)
    if scaler_in is None:
        scaler = StandardScaler()
        fit_mode = True
    else:
        scaler = scaler_in
        fit_mode = False
    feature_columns = ['Log_Ret', 'ATR', 'Z_Score', 'Vol_Imbalance', 'Hurst', 'Half_life', 'ADF_stat', 'PCA_Res', 'Market_Corr', 'VIX']
    
    # 過濾出成功計算出特徵的欄位，防止錯誤
    valid_cols = [c for c in feature_columns if c in features_df.columns]
    if fit_mode:
        std_matrix = scaler.fit_transform(features_df[valid_cols])
    else:
        std_matrix = scaler.transform(features_df[valid_cols])
    
    final_standardized_df = pd.DataFrame(std_matrix, index=features_df.index, columns=valid_cols)
    print(f"特徵矩陣建置完成！維度: {final_standardized_df.shape}")
    
    return final_standardized_df, scaler


In [6]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score

def dynamic_hdbscan_clustering(features_df, sector_map, min_cluster_size=3):
    """
    HDBSCAN 動態分群 (板塊內層過濾)
    自動標記離群值 (Label = -1) 並輸出品質驗證指標。
    """
    features_df = features_df.copy()
    features_df['sector'] = features_df.index.map(lambda x: sector_map.get(x, 'Unknown'))
    feature_cols = [c for c in features_df.columns if c != 'sector']
    
    cluster_labels = pd.Series(index=features_df.index, dtype=int)
    cluster_labels[:] = -1
    
    global_offset = 0
    all_metrics = []
    
    for sector, group in features_df.groupby('sector'):
        if len(group) < min_cluster_size * 2:
            continue
            
        X = group[feature_cols].values
        
        # 實作: HDBSCAN 演算法
        clusterer = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom')
        labels = clusterer.fit_predict(X)
        
        # 自動忽略 RuntimeWarning 發生的輪廓計算
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            valid_mask = labels != -1
            if len(set(labels[valid_mask])) > 1:
                try:
                    sil = silhouette_score(X[valid_mask], labels[valid_mask])
                    ch = calinski_harabasz_score(X[valid_mask], labels[valid_mask])
                    all_metrics.append({'sector': sector, 'silhouette': sil, 'calinski_harabasz': ch})
                except:
                    pass
            
        new_labels = []
        for l in labels:
            if l == -1:
                new_labels.append(-1)
            else:
                new_labels.append(l + global_offset)
                
        cluster_labels.loc[group.index] = new_labels
        if len(set(labels)) > 1:
            global_offset += max(labels) + 1
            
    if all_metrics:
        avg_sil = np.mean([x['silhouette'] for x in all_metrics])
        avg_ch = np.mean([x['calinski_harabasz'] for x in all_metrics])
        # print(f"  [Cluster Quality] Avg Silhouette: {avg_sil:.3f}, Avg CH Index: {avg_ch:.1f}")
        
    return cluster_labels


import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint

def test_single_coint(a, b, sector, price_window, p_threshold, min_len):
    series_a = price_window.get(a)
    series_b = price_window.get(b)
    
    if series_a is None or series_b is None:
        return None
        
    aligned = pd.concat([series_a, series_b], axis=1, join='inner').dropna()
    if len(aligned) < min_len:
        return None
        
    y = aligned.iloc[:, 0]
    x = aligned.iloc[:, 1]
    
    try:
        x_const = sm.add_constant(x)
        res = sm.OLS(y, x_const).fit()
        beta = res.params.iloc[1]
        
        score, pvalue, _ = coint(y, x, maxlag=1)
        
        # 強制過濾掉不合理 Beta，保護保證金不會因過度不對等爆倉
        if pvalue < p_threshold and 0.5 <= beta <= 2.0:
            return {
                'stock_a': a,
                'stock_b': b,
                'sector': sector,
                'p_value': pvalue,
                'beta': beta
            }
    except:
        pass
        
    return None
        
    aligned = pd.concat([series_a, series_b], axis=1, join='inner').dropna()
    if len(aligned) < min_len: return None
        
    y, x = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    try:
        x_const = sm.add_constant(x)
        res = sm.OLS(y, x_const).fit()
        beta = res.params.iloc[1]
        score, pvalue, _ = coint(y, x, maxlag=1)
        
        # 強制過濾掉不合理 Beta，保護保證金不會因過度不對等爆倉
        if pvalue < p_threshold and 0.5 <= beta <= 2.0:
            return {'stock_a': a, 'stock_b': b, 'sector': sector, 'p_value': pvalue, 'beta': beta}
    except:
        pass
    return None


def select_pairs_with_hdbscan(price_window, features_df, sector_map, top_ns, coint_pval=COINT_P_VALUE):
    if isinstance(top_ns, int):
        top_ns = [top_ns]
    """
    雙層過濾機制：
    第一層 HDBSCAN 動態群集
    第二層 Engle-Granger Cointegration
    """
    labels = dynamic_hdbscan_clustering(features_df, sector_map)
    unique_clusters = set(labels) - {-1}
    candidates = []
    
    for cid in unique_clusters:
        cluster_tickers = labels[labels == cid].index.tolist()
        if len(cluster_tickers) >= 2:
            s_map = features_df.loc[cluster_tickers[0], 'sector'] if 'sector' in features_df.columns else sector_map.get(cluster_tickers[0])
            for a, b in combinations(cluster_tickers, 2):
                candidates.append((a, b, s_map))
                
    if not candidates:
        return {n: [] for n in top_ns}
        
    min_len = len(price_window) * 0.8
    norm = price_window / price_window.iloc[0]
    
    coint_results = Parallel(n_jobs=-1, batch_size='auto')(
        delayed(test_single_coint)(a, b, sector, price_window, coint_pval, min_len)
        for a, b, sector in candidates
    )
    
    passed = [res for res in coint_results if res is not None]
    if not passed:
        return {n: [] for n in top_ns}
        
    for p in passed:
        try:
            # [修正點] 依照文獻對齊：Spread = Price_A - Beta * Price_B
            beta = p['beta']
            a_norm = norm[p['stock_a']]
            b_norm = norm[p['stock_b']]
            
            # 計算共整價差
            spread = a_norm - beta * b_norm
            
            # SSD 為價差的平方和
            p['ssd'] = (spread**2).sum()
            
            # 計算過零率 (均值回歸特徵)
            centered = spread - spread.mean()
            zero_crossings = ((centered.shift(1) * centered) < 0).sum()
            p['zero_cross'] = zero_crossings
        except:
            p['ssd'] = np.inf
            p['zero_cross'] = 0
            
    # 依照 SSD 排序並選出 Top-N
    df_passed = pd.DataFrame(passed)
    if df_passed.empty:
        return {n: [] for n in top_ns}
        
    df_passed = df_passed.sort_values('ssd')
    return {n: df_passed.head(n).to_dict('records') for n in top_ns}

# 第三階段：強化學習環境定義 (The MDP Framework)

將配對交易定義為 MDP，實作符合 Gymnasium 標準的環境封裝。代理人將學會如何根據當前 Z-Score 與 VIX 做出決策。

### 參考文獻：
- Brockman, G., et al. (2016). *OpenAI Gym*.

In [7]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

class PairsTradingEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self, data_df, transaction_cost=0.0029, stop_loss_pct=-0.05, beta=1.0, ablation_vix=False):
        super(PairsTradingEnv, self).__init__()
        
        self.df = data_df.reset_index(drop=True)
        self.max_steps = len(self.df) - 1
        
        self.tc = transaction_cost
        self.stop_loss_pct = stop_loss_pct
        self.beta = beta
        self.ablation_vix = ablation_vix
        
        self.action_space = spaces.Discrete(3)
        self.obs_dim = 5 
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )
        
        self.current_step = 0
        self.current_pos = 0       
        self.entry_price_a = 0.0
        self.entry_price_b = 0.0
        self.total_reward = 0.0
        self.pnl_history = [0.0]
        
        self.cumulative_pnl = 0.0
        self.peak_pnl = 0.0
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.current_pos = 0
        self.entry_price_a = 0.0
        self.entry_price_b = 0.0
        self.total_reward = 0.0
        self.pnl_history = [0.0]
        
        self.cumulative_pnl = 0.0
        self.peak_pnl = 0.0
        return self._get_obs(), {}

    def _get_obs(self):
        row = self.df.iloc[self.current_step]
        unrealized_pnl = 0.0
        if self.current_pos != 0 and self.entry_price_a > 0:
            current_pa, current_pb = row['price_a'], row['price_b']
            if not pd.isna(current_pa) and not pd.isna(current_pb):
                unrealized_pnl = self.current_pos * (
                    (current_pa - self.entry_price_a)/self.entry_price_a - 
                    self.beta * (current_pb - self.entry_price_b)/self.entry_price_b
                )
        
        vix_val = 0.0 if self.ablation_vix else (row['vix'] if not pd.isna(row['vix']) else 20.0)
        zscore = row['z_score'] if not pd.isna(row['z_score']) else 0.0
        hl = row['half_life'] if not pd.isna(row['half_life']) else 15.0
        
        obs = np.array([zscore, vix_val, hl, float(self.current_pos), unrealized_pnl], dtype=np.float32)
        return obs

    def step(self, action):
        target_pos = 0
        if action == 1: target_pos = 1
        elif action == 2: target_pos = -1

        row = self.df.iloc[self.current_step]
        current_pa = row['price_a']
        current_pb = row['price_b']
        
        # [Survivorship Bias Fix] Detect Delisting
        is_delisted = pd.isna(current_pa) or pd.isna(current_pb) or current_pa <= 0 or current_pb <= 0
        if is_delisted:
            target_pos = 0
            
        step_pnl = 0.0
        unrealized_pnl = 0.0
        
        if self.current_pos != 0 and self.current_step > 0:
            prev_row = self.df.iloc[self.current_step - 1]
            prev_pa, prev_pb = prev_row['price_a'], prev_row['price_b']
            
            if is_delisted:
                unrealized_pnl = self.stop_loss_pct * 2.0
                step_pnl = unrealized_pnl
            else:
                try:
                    step_pnl = self.current_pos * ((current_pa - prev_pa) / prev_pa - self.beta * (current_pb - prev_pb) / prev_pb)
                    unrealized_pnl = self.current_pos * ((current_pa - self.entry_price_a) / self.entry_price_a - self.beta * (current_pb - self.entry_price_b) / self.entry_price_b)
                except:
                    step_pnl, unrealized_pnl = 0.0, 0.0

        forced_stop_loss = False
        if self.current_pos != 0 and (unrealized_pnl <= self.stop_loss_pct or is_delisted):
            forced_stop_loss = True
            target_pos = 0  
            
        trade_executed = 0
        if target_pos != self.current_pos:
            trades_needed = abs(target_pos - self.current_pos)
            trade_penalty = trades_needed * self.tc * (1 + self.beta)
            step_pnl -= trade_penalty
            if target_pos != 0:
                trade_executed = 1
            
            if target_pos != 0 and not is_delisted:
                self.entry_price_a = current_pa
                self.entry_price_b = current_pb
            else:
                self.entry_price_a, self.entry_price_b, unrealized_pnl = 0.0, 0.0, 0.0

        self.current_pos = target_pos
        self.pnl_history.append(step_pnl)
        
        # 累積資產高峰追蹤
        self.cumulative_pnl += step_pnl
        if self.cumulative_pnl > self.peak_pnl:
            self.peak_pnl = self.cumulative_pnl
        current_drawdown = self.peak_pnl - self.cumulative_pnl
        
        if len(self.pnl_history) > 10:
            rolling_std = np.std(self.pnl_history[-20:]) + 1e-6
            reward = step_pnl / rolling_std
        else:
            reward = step_pnl
            
        # 1. MDD 平方懲罰 (壓制無腦抱單)
        if current_drawdown > 0.02:
            reward -= (current_drawdown ** 2) * 100.0 
            
        # 2. 強力停損扣分 (-5.0) 
        if forced_stop_loss:
            reward -= 5.0
            
        self.current_step += 1
        terminated = bool(self.current_step >= self.max_steps)
        info = {
            'step_pnl': step_pnl, 'unrealized_pnl': unrealized_pnl,
            'forced_stop_loss': forced_stop_loss, 'current_position': self.current_pos,
            'delisted': is_delisted, 'trade_executed': trade_executed
        }
        return self._get_obs(), float(np.clip(reward, -10.0, 10.0)), terminated, False, info

    def render(self, mode='human'):
        pass


# 第四階段：混合獎勵設計與風險控管 (Hybrid Reward Engineering)

設計包含增量損益、換倉代價與停損懲罰的混合獎勵機制，這部分已實作在上述環境類別中，此處進行環境驗證。

### 參考文獻：
- Lucarelli, G., et al. (2019). *A deep reinforcement learning approach for automated cryptocurrency trading*.

In [8]:
# =======================================================
# 驗證單元：環境冒煙測試 (Smoke Test with Dummy Data)
# =======================================================
print('🧪 Starting environment sanity check...')

# 建立模擬數據
dates = pd.date_range('2023-01-01', periods=100)
dummy_df = pd.DataFrame({
    'price_a': 100 + np.cumsum(np.random.normal(0, 1, 100)),
    'price_b': 100 + np.cumsum(np.random.normal(0, 1, 100)),
    'z_score': np.random.normal(0, 1, 100),
    'vix': 20 + np.random.normal(0, 2, 100),
    'half_life': [20.0] * 100
}, index=dates)

# 實例化環境
env_test = PairsTradingEnv(data_df=dummy_df, beta=1.0)
obs, info = env_test.reset()
print(f'✅ Reset successful. Initial Observation: {obs}')

# 執行一個隨機動作並檢查 reward
action = 1 # Long Spread
next_obs, reward, terminated, truncated, info = env_test.step(action)
print(f'✅ Step successful. Reward: {reward:.4f}, Position: {info["current_position"]}')
print('🚀 Sanity check passed! Environment is ready for RL training.')


🧪 Starting environment sanity check...
✅ Reset successful. Initial Observation: [-1.5638286 24.839676  20.         0.         0.       ]
✅ Step successful. Reward: -0.0058, Position: 1
🚀 Sanity check passed! Environment is ready for RL training.


# 第五階段：PPO 模型訓練與樣本外實證 (PPO Evaluation)

採用 PPO 演算法並在樣本外資料 (Out-of-Sample) 上測試模型魯棒性，確保避險邏輯的有效性。

### 參考文獻：
- Schulman, J., et al. (2017). *Proximal policy optimization algorithms*.

In [11]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import gc
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv

def safe_half_life(ts):
    if len(ts) < 30: return 15.0
    z_lag = np.roll(ts, 1)
    z_lag[0] = 0
    z_ret = ts - z_lag
    z_ret[0] = 0
    z_lag2 = sm.add_constant(z_lag)
    try:
        res = sm.OLS(z_ret[1:], z_lag2[1:]).fit()
        hl = -np.log(2) / res.params[1]
        return hl if (hl > 0 and hl < 100) else 15.0
    except:
        return 15.0

def prepare_rl_features(df, beta=1.0, z_window=60):
    df = df.copy()
    df['spread'] = df['price_a'] - beta * df['price_b']
    
    roll_mean = df['spread'].rolling(window=z_window).mean()
    roll_std = df['spread'].rolling(window=z_window).std()
    df['z_score'] = (df['spread'] - roll_mean) / roll_std
    
    hl_list = []
    last_valid_hl = 15.0 
    
    for i in range(len(df)):
        if i < z_window:
            hl_list.append(last_valid_hl)
        elif i % 20 == 0: 
            window_ts = df['spread'].iloc[i-z_window:i].dropna().values
            current_hl = safe_half_life(window_ts)
            hl_list.append(current_hl)
            last_valid_hl = current_hl
        else:
            hl_list.append(hl_list[-1])
            
    df['half_life'] = hl_list
    return df.ffill().fillna(0)

# =========================================================================
# WALK-FORWARD OPTIMIZATION: 日級資產累加引擎 (Daily Portfolio Manager)
# =========================================================================

# 確保 panel有雙重對齊
price_panel_df = prices_raw.copy()
if 'date' not in price_panel_df.index.names:
    price_panel_df.set_index(['date', 'ticker'], inplace=True)

# 系統級參數設定
# FORMATION_WINDOW = 252
# TRADING_WINDOW = 126
ROLLING_STEP = ROLLING_WINDOW
# INITIAL_CAPITAL = 7000.0
TRANCHE_ALLOCATION = INITIAL_CAPITAL / CAPITAL_TRANCHES

print(f"🚀 啟動多視窗整合回測引擎 (Daily Portfolio Manager)")
print(f"📦 總資金: ${INITIAL_CAPITAL} | 單注: ${TRANCHE_ALLOCATION} | 梯隊滾動: {ROLLING_STEP}天")

all_dates = price_pivot.index
total_length = len(all_dates)

cash = INITIAL_CAPITAL
active_tranches = []
completed_tranches_count = 0
trade_frequencies = []
historical_dates = []
historical_portfolio_nav = []

failed_due_to_margin = False

# 每日推演主時間軸
for t in range(FORMATION_WINDOW, total_length):
    current_date = all_dates[t]
    
    # 1. 每日掃描，回收過期梯隊資金
    still_active = []
    for tr in active_tranches:
        if current_date >= tr['end_date']:
            final_nav = tr['series'].iloc[-1]
            cash += final_nav
            completed_tranches_count += 1
        else:
            still_active.append(tr)
    active_tranches = still_active

    # --- [新增] 準確計算每日整個系統的總淨值 ---
    daily_total_nav = cash
    for tr in active_tranches:
        latest_val = tr['series'].loc[:current_date]
        if not latest_val.empty:
            daily_total_nav += latest_val.iloc[-1]
            
    # --- [新增] 全系統破產底線檢查 ---
    if daily_total_nav < 1000.0:
        print(f"❌💥 破產警報！系統總資產 (${daily_total_nav:,.2f}) 已低於 $1000 最低維運標準。交易強制永久終止。")
        failed_due_to_margin = True
        break

    # 2. 定期滾動啟動新梯隊
    if (t - FORMATION_WINDOW) % ROLLING_STEP == 0:
        
        # --- [重點修改] 動態提撥總資產的 10% 建倉 ---
        # TRANCHE_ALLOCATION = INITIAL_CAPITAL * 0.10
        
        if cash < TRANCHE_ALLOCATION:
            # 這不是破產，只是這天現金卡在別的部位裡。不停止整個系統，只是這 20 天休息不開新單。
            print(f"⚠️ 現金水位 (${cash:,.2f}) 暫時不足以提撥 10% 資金 (${TRANCHE_ALLOCATION:,.2f})。新梯隊輪空！")
        else:
            train_start_date = all_dates[t - FORMATION_WINDOW].date()
            
        train_start_date = all_dates[t - FORMATION_WINDOW].date()
        train_end_date = all_dates[t].date()
        test_start_date = all_dates[t].date()
        test_end_idx = min(t + TRADING_WINDOW, total_length - 1)
        test_end_date = all_dates[test_end_idx].date()
        
        print(f"\n--- 🔄 新梯隊啟動 | 系統剩餘現金: ${cash:.2f} | 預計執行區間: {test_start_date} ~ {test_end_date} ---")
        
        train_panel = price_panel_df.loc[str(train_start_date):str(train_end_date)]
        train_pivot = price_pivot.loc[str(train_start_date):str(train_end_date)]
        train_vix = vix_features.loc[str(train_start_date):str(train_end_date)]['VIX']
        
        train_features_std, train_scaler = feature_engineering_pipeline(train_panel, train_vix)
        top_pairs_dict = select_pairs_with_hdbscan(train_pivot, train_features_std, sector_map, top_ns=[1])
        top_pairs = top_pairs_dict.get(1, [])
        
        if not top_pairs:
            print("⚠️ 無高品質共整配對，放棄建倉，資金輪空不使用。")
        else:
            cash -= TRANCHE_ALLOCATION 
            
            best_pair = top_pairs[0]
            stk_a, stk_b, beta_coef = best_pair['stock_a'], best_pair['stock_b'], best_pair['beta']
            print(f"✅ 款項抵扣 ${TRANCHE_ALLOCATION}，選定股票對: {stk_a} vs {stk_b} (Beta: {beta_coef:.4f})")
            
            rl_train_raw = pd.DataFrame({'price_a': train_pivot[stk_a], 'price_b': train_pivot[stk_b], 'vix': train_vix}).dropna()
            rl_train_df = prepare_rl_features(rl_train_raw, beta_coef)
            env_train = PairsTradingEnv(data_df=rl_train_df, transaction_cost=TRANSACTION_COST, beta=beta_coef)
            vec_env_train = DummyVecEnv([lambda: env_train])
            
            model = PPO('MlpPolicy', vec_env_train, verbose=0, learning_rate=0.0003, gamma=0.99)
            model.learn(total_timesteps=30000)
            
            ext_test_start = pd.to_datetime(test_start_date) - pd.Timedelta(days=120)
            test_pivot = price_pivot.loc[ext_test_start:str(test_end_date)]
            test_vix = vix_features.loc[ext_test_start:str(test_end_date)]['VIX']
            
            rl_test_raw = pd.DataFrame({'price_a': test_pivot.get(stk_a, pd.Series(dtype=float)), 'price_b': test_pivot.get(stk_b, pd.Series(dtype=float)), 'vix': test_vix})
            rl_test_full = prepare_rl_features(rl_test_raw, beta_coef)
            actual_test_df = rl_test_full[rl_test_full.index >= pd.to_datetime(test_start_date)].reset_index()
            
            if len(actual_test_df) >= 10:
                env_test = PairsTradingEnv(data_df=actual_test_df, transaction_cost=TRANSACTION_COST, beta=beta_coef)
                vec_env_test = DummyVecEnv([lambda: env_test])
                
                obs = vec_env_test.reset()
                dones = [False]
                
                tranche_nav = TRANCHE_ALLOCATION
                nav_records = [tranche_nav] # 第一天塞入本金佔位，對齊 127 天的長度
                tranche_dates = actual_test_df['date'].tolist()
                trades_count = 0
                
                while not dones[0]:
                    action, _ = model.predict(obs, deterministic=True)
                    obs, rewards, dones, infos = vec_env_test.step(action)
                    trades_count += infos[0].get('trade_executed', 0)
                    
                    tranche_nav += infos[0].get('step_pnl', 0.0) * tranche_nav 
                    nav_records.append(tranche_nav)
                        
                trade_frequencies.append(trades_count)
                
                daily_series = pd.Series(nav_records, index=tranche_dates)
                active_tranches.append({'end_date': tranche_dates[-1], 'series': daily_series})
                del env_test, vec_env_test
            
            del env_train, vec_env_train, model
            gc.collect()

    # 3. 每日掃描：計算總資產 NAV_t
    historical_dates.append(current_date)
    historical_portfolio_nav.append(daily_total_nav)

print("\n🎉 回測迴圈結束！")

# =========================================================================
# 可視化的自動跟隨更新 (對標 SPY) 
# =========================================================================

if len(historical_dates) > 0 and not failed_due_to_margin:
    import yfinance as yf
    print("Fetching SPY benchmark...")
    spy_df = yf.download("SPY", start=str(historical_dates[0].date()), end=str(historical_dates[-1].date()), progress=False)
    spy_close = spy_df['Close'].squeeze()
    spy_close.index = pd.to_datetime(spy_close.index).tz_localize(None)

    results_df = pd.DataFrame({'Date': historical_dates, 'Agent_NAV': historical_portfolio_nav}).set_index('Date')
    
    spy_aligned = spy_close.reindex(results_df.index).ffill()
    spy_initial = spy_aligned.iloc[0]
    results_df['SPY_NAV'] = (spy_aligned / spy_initial) * INITIAL_CAPITAL
    
    results_df['Agent_Peak'] = results_df['Agent_NAV'].cummax()
    results_df['Drawdown'] = (results_df['Agent_NAV'] - results_df['Agent_Peak']) / results_df['Agent_Peak']
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, row_heights=[0.7, 0.3],
                        subplot_titles=('Rolling Tranches Portfolio NAV vs S&P 500', 'Strategy Drawdown Analysis (%)'))

    fig.add_trace(go.Scatter(x=results_df.index, y=results_df['Agent_NAV'], name='Portfolio NAV ($)', line=dict(color='indigo', width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=results_df.index, y=results_df['SPY_NAV'], name='SPY Benchmark ($)', line=dict(color='gray', dash='dash')), row=1, col=1)
    fig.add_trace(go.Scatter(x=results_df.index, y=results_df['Drawdown']*100, name='Drawdown (%)', fill='tozeroy', line=dict(color='red', width=1)), row=2, col=1)

    fig.update_layout(title=f'Rolling Tranches Strategy (Initial: ${INITIAL_CAPITAL})', height=800, template='plotly_white')
    fig.show()

    total_return = (results_df['Agent_NAV'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL
    
    print("\n📊 【滾動梯隊資金管理 - 最終總結結算】 📊")
    print(f"🏁 最終資金餘額: ${results_df['Agent_NAV'].iloc[-1]:,.2f}")
    print(f"📈 累計淨報酬率: {total_return * 100:.2f}%")
    print(f"📉 最大回撤 (MDD): {results_df['Drawdown'].min() * 100:.2f}%")
    print(f"🔄 成功執行退歸的梯隊: {completed_tranches_count} 梯")
    print(f"🛒 全期間總交易次數: {sum(trade_frequencies)} 次\n")


🚀 啟動多視窗整合回測引擎 (Daily Portfolio Manager)
📦 總資金: $10000 | 單注: $1000.0 | 梯隊滾動: 20天

--- 🔄 新梯隊啟動 | 系統剩餘現金: $10000.00 | 預計執行區間: 2001-01-02 ~ 2001-07-03 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 444 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (441, 10)
✅ 款項抵扣 $1000.0，選定雙星: JNJ vs SYK (Beta: 0.7312)

--- 🔄 新梯隊啟動 | 系統剩餘現金: $9000.00 | 預計執行區間: 2001-01-31 ~ 2001-08-01 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 446 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (440, 10)
✅ 款項抵扣 $1000.0，選定雙星: BRK-B vs MET (Beta: 1.0313)

--- 🔄 新梯隊啟動 | 系統剩餘現金: $8000.00 | 預計執行區間: 2001-03-01 ~ 2001-08-29 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 446 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (442, 10)
✅ 款項抵扣 $1000.0，選定雙星: PNC vs MET (Beta: 1.5482)

--- 🔄 新梯隊啟動 | 系統剩餘現金: $7000.00 | 預計執行區間: 2001-03-29 ~ 2001-10-03 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 446 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (444, 10)
✅ 款項抵扣 $1000.0，選定雙星: MSFT vs ZBRA (Beta: 0.6844)

--- 🔄 新梯隊啟動 | 系統剩餘現金: $6000.00 | 預計執行區間: 2001-04-27 ~ 2001-10-31 ---
啟動特徵工程 Pipeline...
啟動多執行緒處理 446 檔股票之時間序列與 ADF 微觀特徵...
特徵矩陣建置完成！維度: (4


📊 【滾動梯隊資金管理 - 最終總結結算】 📊
🏁 最終資金餘額: $2,645.94
📈 累計淨報酬率: -73.54%
📉 最大回撤 (MDD): -75.15%
🔄 成功執行退歸的梯隊: 314 梯
🛒 全期間總交易次數: 583 次

